In [ ]:
    ############    #############   Structured logging   #############   ##############   

 =>  Phase: P0 -- FDE Foundations & Engineering Baseline
 =>  Topic: 0.1 Production Python
 =>  Week:  Week 1
 =>  Track: Core

 =>  Sections to fill in:
       1. Theory
       2. Diagram(s) / flowchart(s) (images/ folder)  -- only where the concept needs one
       3. Command / config reference (if applicable)
       4. Runnable code demo(s)
       5. Hands-on lab checklist
       6. Common pitfalls / notes


In [ ]:
    ############    #############   Structured Logging   #############   ##############   

 =>  Plain text logs ('User 42 logged in') are hard to search/aggregate at scale.
       Structured (JSON) logs are queryable: filter by user_id, trace_id, status, latency.

 =>  A correlation/request ID threaded through every log line for a request lets you
       reconstruct the full story of one request across services.


In [ ]:
import logging
import json
import uuid

class JsonFormatter(logging.Formatter):
    def format(self, record: logging.LogRecord) -> str:
        payload = {
            "level": record.levelname,
            "message": record.getMessage(),
            "logger": record.name,
            "request_id": getattr(record, "request_id", None),
        }
        return json.dumps(payload)

logger = logging.getLogger("fde-academy")
logger.setLevel(logging.INFO)
handler = logging.StreamHandler()
handler.setFormatter(JsonFormatter())
logger.handlers = [handler]

request_id = str(uuid.uuid4())
logger.info("request started", extra={"request_id": request_id})
logger.info("db query executed", extra={"request_id": request_id})


In [ ]:
 =>  Both log lines carry the same request_id -- in a real deployment this is set once
       (e.g. via FastAPI middleware using a contextvar) and every downstream log call
       inherits it, so a log aggregator (Grafana/ELK/Datadog) can group an entire request's
       logs together.


In [ ]:
    ############    #############   Hands-on Lab Checklist   #############   ##############   

 =>  [ ] Add FastAPI middleware that generates a request_id per request (or reads one from
           an incoming X-Request-ID header) and stores it in a contextvars.ContextVar.

 =>  [ ] Update JsonFormatter to pull request_id from that contextvar automatically, so
           callers don't need to pass extra={...} manually every time.


In [ ]:
    ############    #############   Common Pitfalls   #############   ##############   

 =>  Logging secrets/PII (passwords, full card numbers, raw prompts containing user data)
       in plain text -- structured logs still need the same redaction discipline.

 =>  Using print() instead of the logging module in production code -- print() can't be
       filtered by level, routed to a formatter, or shipped to a log aggregator cleanly.
